# GLiNER large -- helio-pii multi-label eval

Loads a GLiNER checkpoint, drives it with the `configs/taxonomy.json`
zero-shot prompts, applies each canonical label's own threshold from
`configs/thresholds.json`, and scores it against the
`data/test_harness.jsonl` multi-label classification harness.

The harness is split once into a **tune** half (used to pick thresholds,
or to choose between checkpoints) and a **holdout** half (used only to
report the final number) -- any metric computed on the same data a
threshold was picked from overstates how well it'll actually do.

In [ ]:
import json
import sys

import pandas as pd
from gliner import GLiNER

sys.path.append("../src")
from taxonomy import load_taxonomy
from metrics import evaluate, sweep_thresholds
from harness import load_harness, stratified_split

taxonomy = load_taxonomy("../configs/taxonomy.json", "../configs/thresholds.json")
taxonomy.thresholds

## Try the format validator (no model needed)

`taxonomy.is_valid(canonical_label, text)` checks a label's exact flagged
text against its `validation_regex` (from `configs/taxonomy.json`) plus a
checksum for `npi`/`credit_card`. Labels with no fixed shape (`name`,
`diagnosis`, ...) always pass -- there's nothing to check them against.
Run this cell any time you change a `validation_regex` to sanity-check it
before re-running the whole harness.

In [ ]:
examples = [
    ("email", "john.smith@email.com"),
    ("email", "contact via email"),  # not a real address -> should fail
    ("phone", "(212) 555-0199"),
    ("phone", "call the office"),  # no digits -> should fail
    ("ssn", "123-45-6789"),
    ("ssn", "123456789"),  # right length, wrong shape -> should fail
    ("npi", "1245319599"),  # passes the CMS checksum
    ("npi", "1234567890"),  # 10 digits but fails the checksum -> should fail
    ("name", "John Smith"),  # no validator for this label -> always passes
]
for canonical, text in examples:
    print(f"{taxonomy.is_valid(canonical, text)!s:6s} is_valid({canonical!r}, {text!r})")

In [ ]:
import torch

MODEL_ID = "urchade/gliner_large-v2.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GLiNER.from_pretrained(MODEL_ID).to(device)
print(f"loaded {MODEL_ID} on {device}")

In [ ]:
docs = load_harness("../data/test_harness.jsonl")
gold_label_sets = [d["labels"] for d in docs]

tune_docs, holdout_docs = stratified_split(docs, holdout_frac=0.3, seed=0)
tune_ids = {d["id"] for d in tune_docs}
holdout_ids = {d["id"] for d in holdout_docs}
print(f"{len(docs)} docs total: {len(tune_docs)} tune / {len(holdout_docs)} holdout")

In [ ]:
def predict_spans(model, text: str) -> list[dict]:
    """Raw span hits with each prompt's own canonical label attached, for
    eyeballing what actually fired. GLiNER's labels ARE the prompt strings,
    so multiple prompt_labels can fire on the same canonical category. Takes
    `model` explicitly (rather than closing over a global) so the same
    taxonomy/prompts can be run through more than one checkpoint for an
    A/B comparison further down."""
    raw = model.predict_entities(text, taxonomy.all_prompts, threshold=taxonomy.score_floor, multi_label=True)
    for e in raw:
        e["canonical"] = taxonomy.prompt_to_canonical[e["label"]]
    return raw


def predict_canonical_scores(model, text: str) -> dict[str, float]:
    """Doc-level score per canonical label, before any threshold is applied:
    the max prompt score seen anywhere in the text for that label, among
    spans that pass that label's format check (taxonomy.is_valid), if it
    has one -- e.g. a span the model calls "email" but that has no @ and
    domain never counts, no matter how confident the model was. Kept
    separate from thresholding so the same scores can be reused to sweep
    thresholds afterwards, without re-running the model."""
    best: dict[str, float] = {}
    for e in predict_spans(model, text):
        if not taxonomy.is_valid(e["canonical"], e["text"]):
            continue
        c = e["canonical"]
        if e["score"] > best.get(c, -1.0):
            best[c] = e["score"]
    return best


def labels_from_thresholds(scores: dict[str, float], thresholds: dict[str, float]) -> list[str]:
    """A canonical label with no span clearing its threshold is simply
    absent -- that's the "non-PII" case, no extra work needed."""
    return [c for c, score in scores.items() if score >= thresholds.get(c, taxonomy.default_threshold)]


def labels_from_scores(scores: dict[str, float]) -> list[str]:
    return labels_from_thresholds(scores, taxonomy.thresholds)


# Regex backstop: catches obvious PII the model missed entirely, unioned in
# rather than replacing the model's predictions -- mirrors
# scala/src/main/scala/GlinerPiiScorer.scala's regexLabels, keep both in
# sync if you change one. The Luhn gate on credit_card is applied here
# directly (not through taxonomy.is_valid's opt-in checksum) because a
# backstop only ever ADDS a finding, so a strict checksum can't make
# recall worse than not having a backstop at all -- unlike gating the
# model's own detections, which is exactly what caused the regression
# where credit_card/ssn/npi collapsed to 0.
import re

from validators import credit_card_checksum_ok

BACKSTOP_PATTERNS = {
    "email": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "phone": r"(?:\(\d{3}\)\s?|\d{3}[-.\s])\d{3}[-.\s]\d{4}",
    "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
    "credit_card": r"\b(?:\d[ -]?){13,16}\b",
}


def apply_backstop(text: str, predicted_labels: list[str]) -> list[str]:
    have = set(predicted_labels)
    for canonical, pattern in BACKSTOP_PATTERNS.items():
        if canonical in have:
            continue
        for m in re.finditer(pattern, text):
            candidate = m.group()
            if not taxonomy.is_valid(canonical, candidate):
                continue
            if canonical == "credit_card" and not credit_card_checksum_ok(candidate):
                continue
            have.add(canonical)
            break
    return list(have)


def finalize_predictions(text: str, scores: dict[str, float], thresholds: dict[str, float]) -> list[str]:
    """What actually ships: threshold the model's (already format-checked)
    scores, then union in anything the regex backstop catches."""
    return apply_backstop(text, labels_from_thresholds(scores, thresholds))

In [ ]:
import time

from tqdm.auto import tqdm


def score_harness(model, docs: list[dict]) -> list[dict]:
    """Run one checkpoint over the whole harness once, returning the raw
    per-doc canonical scores -- reused for the baseline report, threshold
    sweeping, and holdout evaluation, so the model only runs once per doc."""
    scores = []
    start = time.monotonic()
    for d in tqdm(docs, desc="scoring documents", unit="doc"):
        scores.append(predict_canonical_scores(model, d["text"]))
    elapsed = time.monotonic() - start
    print(f"scored {len(docs)} docs in {elapsed:.1f}s ({elapsed / len(docs):.2f}s/doc)")
    return scores


canonical_scores = score_harness(model, docs)
predictions = [finalize_predictions(d["text"], s, taxonomy.thresholds) for d, s in zip(docs, canonical_scores)]
for d, pred in zip(docs, predictions):
    print(f"{d['id']:24s} gold={d['labels']!s:30s} pred={pred}")

### If a label's numbers move in a surprising way

`validation_debug` shows every raw span the model found for one
canonical label, before thresholding -- whether it passed `is_valid`
and why, and whether the document is actually gold-positive for that
label. Use this instead of guessing when a `validation_regex` or
`validation_checksum` change makes a label's precision/recall jump.

In [ ]:
def validation_debug(model, docs: list[dict], canonical: str, limit: int = 15) -> None:
    shown = 0
    for d in docs:
        for e in predict_spans(model, d["text"]):
            if e["canonical"] != canonical:
                continue
            valid = taxonomy.is_valid(canonical, e["text"])
            gold = canonical in d["labels"]
            print(f"{d['id']:24s} score={e['score']:.2f} valid={valid!s:5s} gold={gold!s:5s} text={e['text']!r}")
            shown += 1
            if shown >= limit:
                return


for label in ["credit_card", "ssn", "npi", "email", "phone"]:
    print(f"=== {label} ===")
    validation_debug(model, docs, label)

## Baseline: default (untuned) thresholds

`configs/thresholds.json` is tagged `"initial-operating-profile"` -- these
numbers are a starting point, not a verdict.

In [ ]:
rows, micro, macro = evaluate(gold_label_sets, predictions, taxonomy.canonical_names)
per_label_df = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)
per_label_df

In [ ]:
print("micro:", micro)  # dominated by frequent labels
print("macro:", macro)  # every label weighted equally -- watch this for rare PHI labels

In [ ]:
# The harness tags each row with scenario_type/noise_type/polarity -- break
# exact-match accuracy down by scenario_type to see where it's weakest
# (e.g. boundary_ambiguous vs. near_miss_negative).
harness_df = pd.DataFrame(docs)
harness_df["predicted"] = predictions
harness_df["exact_match"] = [set(g) == set(p) for g, p in zip(gold_label_sets, predictions)]
harness_df.groupby("scenario_type")["exact_match"].mean()

## Threshold tuning (tune split only)

Sweep each label's threshold using only `tune_docs`' scores -- already
computed above, no extra model calls. The result is evaluated on
`holdout_docs` in the next cell, which is the number to actually trust.

In [ ]:
def split_scores(
    scores: list[dict], gold: list[list[str]], ids: set[str]
) -> tuple[list[dict], list[list[str]], list[dict]]:
    keep = [i for i, d in enumerate(docs) if d["id"] in ids]
    return [scores[i] for i in keep], [gold[i] for i in keep], [docs[i] for i in keep]


tune_scores, tune_gold, _ = split_scores(canonical_scores, gold_label_sets, tune_ids)
holdout_scores, holdout_gold, holdout_docs_ordered = split_scores(canonical_scores, gold_label_sets, holdout_ids)

tuned_rows = sweep_thresholds(tune_gold, tune_scores, taxonomy.canonical_names)
tuned_thresholds = {row["label"]: row["threshold"] for row in tuned_rows}
pd.DataFrame(tuned_rows).sort_values("label").reset_index(drop=True)

In [ ]:
holdout_preds = [
    finalize_predictions(d["text"], s, tuned_thresholds) for d, s in zip(holdout_docs_ordered, holdout_scores)
]
holdout_rows, holdout_micro, holdout_macro = evaluate(holdout_gold, holdout_preds, taxonomy.canonical_names)
print("holdout micro (honest, tuned thresholds + regex backstop):", holdout_micro)
print("holdout macro (honest, tuned thresholds + regex backstop):", holdout_macro)

In [ ]:
tuned_config = {
    "schema_version": 1,
    "status": "tuned-on-test-harness",
    "model": MODEL_ID,
    "taxonomy": "configs/taxonomy.json",
    "default_threshold": taxonomy.default_threshold,
    "thresholds": tuned_thresholds,
}
with open("../configs/thresholds.tuned.json", "w") as f:
    json.dump(tuned_config, f, indent=2)
print("wrote ../configs/thresholds.tuned.json -- review, then replace configs/thresholds.json if it looks right")

## Error inspection

The worst-F1 labels from the baseline run, with a few concrete false
positive / false negative examples each -- this is usually more actionable
than the aggregate numbers for deciding whether a `prompt_labels` entry is
too broad (FPs) or too narrow (FNs).

In [ ]:
def label_errors(label: str):
    fps, fns = [], []
    for d, pred in zip(docs, predictions):
        gold, predset = set(d["labels"]), set(pred)
        if label in predset and label not in gold:
            fps.append(d)
        elif label in gold and label not in predset:
            fns.append(d)
    return fps, fns


worst_labels = per_label_df.sort_values("f1").head(3)["label"].tolist()
for label in worst_labels:
    fps, fns = label_errors(label)
    print(f"=== {label}: {len(fps)} false positives, {len(fns)} false negatives ===")
    for d in fps[:3]:
        print(f"  FP [{d['scenario_type']}] {d['id']}: {d['text'][:100]!r}")
    for d in fns[:3]:
        print(f"  FN [{d['scenario_type']}] {d['id']}: {d['text'][:100]!r}")

## A/B: swap the backbone checkpoint

`gliner_multi_pii-v1` is fine-tuned specifically on PII, but it's still a
GLiNER checkpoint: labels are natural-language prompts passed in at
inference time, not a fixed output head, so `taxonomy.all_prompts` --
your own zero-shot prompts (e.g. "medical record number") included --
works against it exactly the same way, no taxonomy changes needed.

To decide fairly: tune model B's thresholds on the *same* tune split, then
compare both models' *holdout* metrics. Comparing at one fixed threshold
(or tuning and comparing on the same data) can just reward whichever model
happens to fit that threshold or that data, not whichever generalizes
better.

This costs a second full pass over the harness (same runtime as model A's
run, reported above). If you're on a memory-constrained GPU, free the
first model first (`del model; torch.cuda.empty_cache()`) before loading
the second.

In [ ]:
MODEL_B_ID = "urchade/gliner_multi_pii-v1"
model_b = GLiNER.from_pretrained(MODEL_B_ID).to(device)
print(f"loaded {MODEL_B_ID} on {device}")

canonical_scores_b = score_harness(model_b, docs)
tune_scores_b, _, _ = split_scores(canonical_scores_b, gold_label_sets, tune_ids)
holdout_scores_b, _, _ = split_scores(canonical_scores_b, gold_label_sets, holdout_ids)

tuned_rows_b = sweep_thresholds(tune_gold, tune_scores_b, taxonomy.canonical_names)
tuned_thresholds_b = {row["label"]: row["threshold"] for row in tuned_rows_b}
holdout_preds_b = [
    finalize_predictions(d["text"], s, tuned_thresholds_b) for d, s in zip(holdout_docs_ordered, holdout_scores_b)
]
holdout_rows_b, holdout_micro_b, holdout_macro_b = evaluate(holdout_gold, holdout_preds_b, taxonomy.canonical_names)

In [ ]:
# The decision: each model got its own thresholds, tuned only on
# tune_docs; these numbers are both from holdout_docs, which neither
# model's thresholds were picked from.
print(f"{MODEL_ID:35s} holdout micro={holdout_micro}")
print(f"{MODEL_B_ID:35s} holdout micro={holdout_micro_b}")
print(f"{MODEL_ID:35s} holdout macro={holdout_macro}")
print(f"{MODEL_B_ID:35s} holdout macro={holdout_macro_b}")

compare_df = pd.DataFrame(holdout_rows).set_index("label")[["precision", "recall", "f1"]].add_suffix("_large_v2_1")
compare_df_b = pd.DataFrame(holdout_rows_b).set_index("label")[["precision", "recall", "f1"]].add_suffix("_multi_pii_v1")
compare_df.join(compare_df_b).sort_index()